# RouteRanker Module 6: Inference and Route Optimization

Use the saved Module 5 models and Module 4 preprocessing manifest to analyze unseen route records, review possible route changes, and compare the current network with an illustrative schedule scenario.

## Run in VS Code

1. Open the RouteRanker Module 6 project folder in VS Code.
2. Select the project's Python kernel.
3. If needed, install dependencies in the project terminal: `python -m pip install -r requirements-notebook.txt`.
4. Run cells from top to bottom. The workflow writes its outputs under `results/`, `data/simulation/`, `reports/`, and `visualizations/`.

The sample unseen input is the held-out Module 4 validation split. A raw feature input is transformed with the saved training medians, log transforms, robust scaler, cyclic features, and one-hot categories. No preprocessing step is refit here.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import SVG, display

search_locations = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (p for p in search_locations
     if (p / "models" / "model_index.json").is_file()
     and (p / "data" / "processed" / "module4_preprocessing_manifest.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the RouteRanker Module 6 folder in VS Code, or set PROJECT_ROOT to that folder.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from route_ranker.module6_workflow import run_module6

UNSEEN_INPUT = PROJECT_ROOT / "data" / "inference" / "unseen_routes.csv"
print(f"Project root: {PROJECT_ROOT}")
print(f"Unseen input: {UNSEEN_INPUT}")


## 1. Check the saved preprocessing and unseen input

In [ ]:
import json
manifest_path = PROJECT_ROOT / "data" / "processed" / "module4_preprocessing_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(f"Prepared source rows: {manifest['rows']['input']:,}")
print(f"Model feature count: {len(manifest['feature_columns'])}")
print(f"Train rows: {manifest['rows']['train']:,}; validation rows: {manifest['rows']['validation']:,}")
display(pd.read_csv(UNSEEN_INPUT).head())


## 2. Run saved-model inference and the optimization workflow

The workflow assigns clusters on the held-out routes and across the network, builds evidence-based review lists, and creates a separate frequency sensitivity dataset.

In [ ]:
summary = run_module6(PROJECT_ROOT, unseen_input=UNSEEN_INPUT)
display(pd.json_normalize(summary, sep="." ).T.rename(columns={0: "value"}))


## 3. Inspect unseen route assignments

In [ ]:
assignments = pd.read_csv(PROJECT_ROOT / "results" / "inference_assignments.csv")
print(f"Unseen routes assigned: {len(assignments):,}")
display(assignments.head(20))


## 4. Review model signals and proposed actions

These are screening suggestions, not declarations that routes are inefficient. Merge and split candidates require additional coverage, stop-sequence, transfer, demand, and capacity checks.

In [ ]:
for filename in [
    "potential_route_review.csv",
    "merge_review_candidates.csv",
    "split_review_candidates.csv",
    "optimization_suggestions.csv",
    "classification_availability.csv",
]:
    path = PROJECT_ROOT / "results" / filename
    print(filename)
    display(pd.read_csv(path).head(25))


## 5. Compare current and simulated network metrics

The sensitivity scenario adds one scheduled trip to each route with high same-city headway and low same-city scheduled-trip count. It does not apply merge or split changes.

In [ ]:
metrics = pd.read_csv(PROJECT_ROOT / "results" / "current_vs_simulated_network_metrics.csv")
display(metrics)


## 6. Visualizations

In [ ]:
for filename in [
    "module6_holdout_cluster_counts.svg",
    "module6_network_scenario_comparison.svg",
]:
    path = PROJECT_ROOT / "visualizations" / filename
    print(filename)
    display(SVG(filename=str(path)))


## Interpretation

- K-Means uses the nearest saved centroid.
- DBSCAN uses the nearest saved core point within `eps`; this is an approximation because DBSCAN has no native prediction method.
- Clusters and outlier signals are not route-quality labels.
- The data describes scheduled service, not passenger counts or actual demand. Random Forest and XGBoost predictions remain unavailable unless valid labeled models are supplied.
- The simulated output is separate from the original network. Verify service coverage, passenger journeys, vehicle capacity, operating costs, and accessibility before making a real schedule change.
